# 따릉이 대여량 예측 — 탐색적 데이터 분석(EDA)

- 목적: 모델 학습 전, 데이터의 분포·품질을 먼저 확인
- 도구: YData Profiling (데이터프레임 한 줄 입력 → EDA 리포트 자동 생성)
- 대상 데이터: 서울 공공자전거 시간별 대여 기록 (2017.12 ~ 2018.11)
- 흐름: 데이터 불러오기 → 리포트 생성 → 리포트 읽기 → 전처리 대상 도출

In [ ]:
# !pip install fg-data-profiling
# !pip install -U ipywidgets

## 1. 라이브러리 불러오기

- pandas: 표 데이터 처리
- ProfileReport: EDA 리포트 생성

In [ ]:
import pandas as pd
from data_profiling import ProfileReport

## 2. 데이터 불러오기

- 파일: SeoulBikeData.csv
- 인코딩: latin-1 (℃ 등 특수문자 포함)
- Date 열: 문자열 → 날짜형 변환

In [ ]:
df = pd.read_csv("../SeoulBikeData.csv", encoding="latin-1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df.head()

## 3. 데이터 크기·구성 확인

- shape: 행·열 개수
- info: 열별 타입·결측 여부
- describe: 수치형 요약 통계

In [ ]:
print("행·열:", df.shape)
df.info()

In [ ]:
df.describe()

## 4. EDA 리포트 생성

- 입력: 데이터프레임 한 줄
- 출력: 6개 섹션 리포트 (Overview · Variables · Correlations · Missing values · Duplicate rows · Sample)
- 확인 방법 두 가지
    - 노트북 안에서 바로 보기: `to_notebook_iframe()`
    - HTML 파일로 저장 후 브라우저에서 열기: `to_file()`

In [ ]:
profile = ProfileReport(df, title="따릉이 EDA")

In [ ]:
# 방법 A — HTML 파일로 저장 후 브라우저에서 열기
profile.to_file("seoul_bike_eda.html")

In [ ]:
# 방법 B — 노트북 안에서 확인 
profile.to_notebook_iframe()

## 5. 리포트에서 확인할 항목

- Overview
    - 행·열 수, 전체 결측·중복 총량
    - Alerts: 데이터 품질 경고 (상수 열 · 높은 상관 · 치우침 등)
- Variables
    - 변수별 분포 · 평균 · 최소 · 최대 · 고유값 수
    - 확인 포인트: Hour 분포(오전·오후 이봉), 대여량 치우침
- Missing values
    - 결측 위치 · 비율
- Correlations
    - 예측 대상(대여량)과 함께 움직이는 변수
- Duplicate rows
    - 중복 행 유무

## 6. 발견 → 전처리 연결

- 리포트에서 확인한 사항을 코드로 직접 검증
- 이후 전처리 단계의 처리 대상 도출

### 6-1. Hour 분포 — 시간대별 대여량

- 확인: 오전·오후 두 번 피크 (이봉 분포)
- 시사점: Hour를 숫자가 아닌 범주형으로 다룰 근거

In [ ]:
df.groupby("Hour")["Rented Bike Count"].mean()

### 6-2. 숫자로 위장한 결측·이상치 점검

- 확인: 각 수치형 열의 최소·최대값
- 시사점: 999 · -1 등 실제값이 아닌 코드가 섞였는지 확인

In [ ]:
num_cols = df.select_dtypes(include="number").columns
df[num_cols].agg(["min", "max"])

### 6-3. 불필요한 열 점검

- 확인: 값이 하나뿐인 열(상수), 열별 고유값 수
- 시사점: 정보가 없는 열은 제거 대상

In [ ]:
df.nunique()

### 6-4. Functioning Day 점검

- 확인: 운영일 여부별 대여량
- 시사점: 미운영일(No)의 대여량이 0인지 확인

In [ ]:
df.groupby("Functioning Day")["Rented Bike Count"].agg(["mean", "min", "max", "count"])

## 정리

- EDA: 모델 학습 전, 데이터의 분포·품질을 확인하는 단계
- YData Profiling: 데이터프레임 한 줄로 리포트 자동 생성
- 리포트에서 확인한 문제 → 다음 단계 전처리의 처리 대상
- 다음 단계: 전처리 (결측 처리 · 인코딩 · Feature Engineering)